In [1]:
import dgl
from dgl.data import TreeGridDataset
import torch
import torch_geometric
import torch_geometric.transforms as T

In [2]:
import argparse
import time
import easydict
from torch.nn import Linear
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

In [3]:
import torch_geometric.utils
from torch_geometric.utils.convert import from_networkx
from torch_geometric.logging import log
import os
import pandas as pd
import glob
import pickle

In [4]:
dataset = TreeGridDataset()
g = dataset[0]

Done loading data from cached files.


In [1]:
#Download file Tree_Grids.pkl from the dataset in https://github.com/Graph-and-Geometric-Learning/D4Explainer. Tree_Grids.pkl is required for the train/val/test splits.

In [5]:
with open('Tree_Grids.pkl', 'rb') as fin:
    adj, features, y_train, y_val, y_test, train_mask, val_mask, test_mask, edge_label_matrix  = pickle.load(fin)

In [6]:
len(adj)

1231

In [7]:
g.ndata["train_mask"] = torch.tensor(train_mask)
g.ndata["val_mask"] = torch.tensor(val_mask)
g.ndata["test_mask"] = torch.tensor(test_mask)

In [8]:
data = torch_geometric.utils.from_dgl(g)
data.x = data.feat
data.y = data.label
data.pop('feat')
data.pop('__orig__')
data.pop('label')
#data

tensor([1, 1, 1,  ..., 0, 0, 0])

In [9]:
data.x = torch.zeros(data.x.shape[0],2)
data

Data(edge_index=[2, 1705], train_mask=[1231], val_mask=[1231], test_mask=[1231], x=[1231, 2], y=[1231])

In [10]:
parser = argparse.ArgumentParser()
args = easydict.EasyDict({
    "dataset": 'TreeGrid',
    #"batch_size": 128,
    # "hidden_channels": 64,
    # "lr": 0.0005,
    "epochs": 2000,
})

In [11]:
parser = argparse.ArgumentParser()
args = easydict.EasyDict({
    "dataset": 'BAShapes',
    #"batch_size": 128,
    #"hidden_channels": 64,
    #"lr": 0.0005,
    "epochs": 2000,
})

In [12]:
device = 'cpu'

In [13]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels,
                             )
        self.conv2 = GCNConv(hidden_channels, out_channels,
                             )

    def forward(self, x, edge_index, edge_weight=None):
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv1(x, edge_index, edge_weight).relu()
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index, edge_weight)
        return x

device = 'cpu'
model = GCN(
    in_channels=data.x.shape[1],
    hidden_channels=16,
    out_channels=2,
).to(device)

optimizer = torch.optim.Adam([
    dict(params=model.conv1.parameters(), weight_decay=5e-4),
    dict(params=model.conv2.parameters(), weight_decay=0)
], lr=0.01)  # Only perform weight-decay on first convolution.


def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    #train_idx = data.y != -1 
    #loss = F.cross_entropy(out[train_idx], data.y[train_idx])
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return float(loss)


@torch.no_grad()
def test():
    model.eval()
    pred = model(data.x, data.edge_index).argmax(dim=-1)

    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        accs.append(int((pred[mask] == data.y[mask]).sum()) / int(mask.sum()))
    return accs


best_val_acc = test_acc = 0
start_patience = patience = 100
times = []
for epoch in range(1, 2000 + 1):
    start = time.time()
    loss = train()
    train_acc, val_acc, tmp_test_acc = test()
    if val_acc > best_val_acc:
        #best_val_acc = val_acc
        test_acc = tmp_test_acc
    if epoch%10==0:
        log(Epoch=epoch, Loss=loss, Train=train_acc, Val=val_acc, Test=test_acc)
    times.append(time.time() - start)

    if (val_acc>best_val_acc):
        print('saving....')
        patience = start_patience
        best_val_acc = val_acc
        print('best acc is', best_val_acc)

        if not os.path.isdir('checkpoint'):
            os.mkdir('checkpoint')
        torch.save(model.state_dict(), './checkpoint/TreeGrids_gcn.pth')
    else:
        patience -= 1
        
    if patience <= 0:
        print('Stopping training as validation accuracy did not improve '
              f'for {start_patience} epochs')
        break   
        #torch.save(model.cpu(), './checkpoint/dblp_gcn.pt')
print(f'Median time per epoch: {torch.tensor(times).median():.4f}s')

saving....
best acc is 0.6178861788617886
Epoch: 010, Loss: 0.6838, Train: 0.5752, Val: 0.6179, Test: 0.6290
Epoch: 020, Loss: 0.6818, Train: 0.5752, Val: 0.6179, Test: 0.6290
Epoch: 030, Loss: 0.6822, Train: 0.5752, Val: 0.6179, Test: 0.6290
Epoch: 040, Loss: 0.6818, Train: 0.5752, Val: 0.6179, Test: 0.6290
Epoch: 050, Loss: 0.6818, Train: 0.5752, Val: 0.6179, Test: 0.6290
Epoch: 060, Loss: 0.6818, Train: 0.5752, Val: 0.6179, Test: 0.6290
Epoch: 070, Loss: 0.6818, Train: 0.5752, Val: 0.6179, Test: 0.6290
Epoch: 080, Loss: 0.6818, Train: 0.5752, Val: 0.6179, Test: 0.6290
Epoch: 090, Loss: 0.6818, Train: 0.5752, Val: 0.6179, Test: 0.6290
Epoch: 100, Loss: 0.6818, Train: 0.5752, Val: 0.6179, Test: 0.6290
Stopping training as validation accuracy did not improve for 100 epochs
Median time per epoch: 0.0020s
